# ⚠️ LEGACY / KHÔNG DÙNG ĐỂ CHỌN CHECKPOINT

Notebook cũ này dùng token-position accuracy, focal loss/rebalance và preprocessing không còn hợp lệ. Hãy chạy `python -m reader.train` để dùng shared answer_start + BPE offsets, decoded QA F1, threshold calibration và full validation. Notebook được giữ lại chỉ để audit lịch sử.

In [ ]:
# 1. Cài đặt các thư viện cần thiết & Tắt cảnh báo thừa
!pip install -q transformers datasets accelerate scikit-learn pandas pyarrow pyvi sentencepiece

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 2. Import các thư viện
import os
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    RobertaTokenizerFast,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    logging as hf_logging
)

# Ẩn các log/warning không cần thiết từ HuggingFace
hf_logging.set_verbosity_error()

In [ ]:
# 3. Định nghĩa các hàm tiện ích xử lý dữ liệu
def find_char_span(context: str, answer: str):
    if not answer or not context:
        return -1, -1
    c_chars = [(char, i) for i, char in enumerate(context) if char not in (' ', '_')]
    a_chars = [char for char in answer if char not in (' ', '_')]
    c_str = ''.join([x[0] for x in c_chars])
    a_str = ''.join(a_chars)
    idx = c_str.find(a_str)
    if idx == -1:
        return -1, -1
    start_char_idx = c_chars[idx][1]
    end_char_idx = c_chars[idx + len(a_str) - 1][1] + 1
    return start_char_idx, end_char_idx

def load_qa_dataset(file_path: str) -> Dataset:
    df = pd.read_parquet(file_path)
    if "context_segmented" in df.columns:
        df = df.drop(columns=["context", "question", "answer_text"], errors="ignore")
        df = df.rename(columns={
            "context_segmented": "context",
            "question_segmented": "question",
            "answer_text_segmented": "answer_text"
        })
    data_dict = {
        "id": df["id"].astype(str).tolist(),
        "context": df["context"].astype(str).tolist(),
        "question": df["question"].astype(str).tolist(),
        "answer_text": df["answer_text"].fillna("").astype(str).tolist(),
        "answer_start": df["answer_start"].fillna(-1).astype(int).tolist(),
    }
    return Dataset.from_dict(data_dict)

def prepare_train_features(examples, tokenizer, max_seq_len=256, doc_stride=85):
    examples["question"] = [q[:150] for q in examples["question"]]
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=max_seq_len,
        stride=doc_stride,
        return_overflowing_tokens=True,
        padding="max_length",
    )
    
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    
    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []
    
    for i in range(len(tokenized_examples["input_ids"])):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        
        sample_index = sample_mapping[i]
        raw_answer = examples["answer_text"][sample_index]
        raw_start = examples["answer_start"][sample_index]
        
        if raw_start == -1 or not raw_answer.strip():
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
            continue
            
        ans_ids = tokenizer.encode(raw_answer, add_special_tokens=False)
        if not ans_ids:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
            continue
            
        n = len(ans_ids)
        found = False
        sequence_ids = tokenized_examples.sequence_ids(i)
        
        context_start = 0
        while context_start < len(sequence_ids) and sequence_ids[context_start] != 1:
            context_start += 1
            
        context_end = len(sequence_ids) - 1
        while context_end >= 0 and sequence_ids[context_end] != 1:
            context_end -= 1
            
        if context_start <= context_end:
            for j in range(context_start, context_end - n + 2):
                if input_ids[j:j+n] == ans_ids:
                    tokenized_examples["start_positions"].append(j)
                    tokenized_examples["end_positions"].append(j + n - 1)
                    found = True
                    break
                    
        if not found:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
            
    return tokenized_examples

import numpy as np
from datasets import Dataset

def rebalance_features(ds: Dataset, cls_ratio: float = 2.0) -> Dataset:
    """Giới hạn feature CLS (no-answer) = cls_ratio * feature có đáp án."""
    cls_id = 0
    has_span = np.array([
        s != cls_id or e != cls_id
        for s, e in zip(ds["start_positions"], ds["end_positions"])
    ])
    span_idx = np.where(has_span)[0]
    cls_idx = np.where(~has_span)[0]
    target_cls = min(len(cls_idx), int(len(span_idx) * cls_ratio))
    keep_cls = np.random.choice(cls_idx, target_cls, replace=False)
    keep = np.concatenate([span_idx, keep_cls])
    np.random.shuffle(keep)
    return ds.select(keep)

def compute_metrics(eval_pred):
    logits_s, logits_e = eval_pred.predictions
    start_pos, end_pos = eval_pred.label_ids
    cls_id = 0
    mask = (start_pos != cls_id) | (end_pos != cls_id)
    if mask.sum() == 0:
        return {"answerable_em": 0.0}
    pred_s = logits_s[mask].argmax(-1)
    pred_e = logits_e[mask].argmax(-1)
    em = (pred_s == start_pos[mask]) & (pred_e == end_pos[mask])
    return {"answerable_em": float(em.mean())}


In [ ]:
# 4. Đọc dữ liệu (Tự động tìm đường dẫn trên Kaggle hoặc Colab)
import os
train_path = "viquad_train_segmented.parquet"
val_path = "viquad_val_segmented.parquet"

if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if "train" in f and f.endswith(".parquet"):
                train_path = os.path.join(root, f)
            elif "val" in f and f.endswith(".parquet"):
                val_path = os.path.join(root, f)

print(f"Train path: {train_path}")
print(f"Val path:   {val_path}")

train_dataset = load_qa_dataset(train_path)
val_dataset = load_qa_dataset(val_path)
print(f"Tập Train: {len(train_dataset)} mẫu, Tập Val: {len(val_dataset)} mẫu")

In [ ]:
# 5. Tải Tokenizer và tiền xử lý dữ liệu
model_name = "vinai/phobert-base-v2"
print(f"Đang tải RobertaTokenizerFast từ {model_name}...")
tokenizer = RobertaTokenizerFast.from_pretrained(model_name)

max_seq_len = 256
doc_stride = 85          # was 85  (match predict.py)

print("Đang tiền xử lý (Tokenize) dữ liệu...")
tokenized_train = train_dataset.map(
    lambda x: prepare_train_features(x, tokenizer, max_seq_len, doc_stride),
    batched=True,
    remove_columns=train_dataset.column_names
)
tokenized_train = rebalance_features(tokenized_train, cls_ratio=0.3)  # ← sửa: 0.3 thay vì 2.0

tokenized_val = val_dataset.map(
    lambda x: prepare_train_features(x, tokenizer, max_seq_len, doc_stride),
    batched=True,
    remove_columns=val_dataset.column_names
)
print(f"Số lượng features sau Tokenize - Train: {len(tokenized_train)}, Val: {len(tokenized_val)}")

In [ ]:
# 6. Huấn luyện (Tăng tốc & Hiện tiến độ liên tục vào bảng)
print(f"Đang khởi tạo mô hình PhoBERT v2...")
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

output_dir = "./results_phobert"

eval_key = "eval_strategy" if hasattr(TrainingArguments("./tmp"), "eval_strategy") else "evaluation_strategy"
kwargs_args = {
    "output_dir": output_dir,
    eval_key: "steps",
    "eval_steps": 500,
    "learning_rate": 5e-5,                    # was 3e-5
    "per_device_train_batch_size": 32,
    "per_device_eval_batch_size": 32,
    "num_train_epochs": 5,                    # was 2
    "warmup_ratio": 0.1,                      # thêm
    "lr_scheduler_type": "cosine",            # thêm
    "weight_decay": 0.01,
    "save_total_limit": 1,
    "logging_steps": 50,
    "save_strategy": "steps",
    "save_steps": 500,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_answerable_em",  # was "loss"
    "greater_is_better": True,                # was False
    "report_to": "none",
    "fp16": torch.cuda.is_available(),
}
training_args = TrainingArguments(**kwargs_args)

# Focal Loss Trainer để xử lý imbalance cực đoan
from torch.nn import functional as F

class FocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels_s = inputs.pop("start_positions")
        labels_e = inputs.pop("end_positions")
        outputs = model(**inputs)
        logits_s, logits_e = outputs.start_logits, outputs.end_logits
        
        ce_s = F.cross_entropy(logits_s, labels_s, reduction="none")
        ce_e = F.cross_entropy(logits_e, labels_e, reduction="none")
        pt_s = torch.exp(-ce_s)
        pt_e = torch.exp(-ce_e)
        focal_s = 0.25 * (1 - pt_s) ** 2 * ce_s
        focal_e = 0.25 * (1 - pt_e) ** 2 * ce_e
        loss = (focal_s + focal_e).mean()
        
        return (loss, outputs) if return_outputs else loss

try:
    trainer = FocalTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )
except TypeError:
    trainer = FocalTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

print("🚀 Bắt đầu quá trình huấn luyện PhoBERT v2...")
trainer.train()
print("✅ Huấn luyện hoàn tất!")

In [ ]:
# 7. Lưu mô hình & nén zip
final_model_dir = "./vinai_phobert-base-v2"
print(f"Đang lưu mô hình tốt nhất vào {final_model_dir}...")
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

!zip -r vinai_phobert-base-v2.zip {final_model_dir}
print("🎉 Đã nén xong! Hãy tải tệp 'vinai_phobert-base-v2.zip' về máy của bạn.")

In [ ]:
# 8. Xác minh sau khi huấn luyện (Verify after training)
from reader.evaluate import evaluate
evaluate(
    model_path="./vinai_phobert-base-v2",
    data_variant="segmented",
    subset_size=100,
    use_cpu=True,
    output_file="./retrained_eval.json"
)
# Target: Answerable EM ≥ 60%